## Avalia??o em holdouts - Amazon Books

Interactions in the 'Amazon Books' dataset are well distributed.
Several users are present during the whole considered period (2014):
<!-- * 37067 users of 190248 (19.484%) occurr in 80.0% or more months. -->
* 2052 users of 16238 (12.637%) occurr in 100% or more months.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('') + '/..')

In [ ]:
from data import ImplicitData, getBucketsHoldouts
from plot_utils import lineplot_recallxholdout, recall_heatmap
from dataset_evaluation_utils import *
from recommenders_implicit import ISGD, RAISGD, RSISGD  # ISGD framework, BISGD,
from eval_implicit import EvaluateHoldouts, EvaluateAndStore, EvalPrequential # EvaluateAndStore para guardar estados do modelo e holdouts, a avalia??o prequencial de ratings implicitos ? opcional, , EvalHoldout

from datetime import datetime
import joblib
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style('whitegrid')

___
## BWT FWT

ACC, BWT, e FWT - Lopez-Paz e Ranzato GEM

In [ ]:
def avg_recall(results_matrix): # Lopez-Paz e Ranzato GEM 2017
    return np.mean( np.diag(results_matrix) )

def compute_BWT(results_matrix): # Lopez-Paz e Ranzato GEM 2017
    BWT = []
    n_checkpoints = results_matrix.shape[0]
    for T in range(1, n_checkpoints): # 1 means holdout 2, 2 means 3, so on
        Rti = results_matrix.iloc[T, 0:T] # get models performances' on previous holdouts
        Rii = np.diag(results_matrix)[0:T] # get models performances' on their closest holdouts (diagonal)
        E = sum( Rti - Rii ) # future models performances' - performances' of models closest to holdouts (diagonal)
        BWT.append( E/T ) # store average BWT for model
    return BWT, np.mean( BWT ) # return BWT and average BWT for all models

def compute_FWT(results_matrix): # D?az-Rodriguez et al. 2018
    upper_tri = results_matrix.to_numpy()[np.triu_indices(results_matrix.shape[0], k=1)]
    return np.mean(upper_tri)

___
# Amazon Books
Small subset, ratings only.  
https://nijianmo.github.io/amazon/index.html


In [ ]:
# importa dataset 'books playlists'
data = pd.read_csv('output/amazonbooks_dump/3rd_sampled_amazon_books.csv')
dataset_name = 'Amazon_Books'
user_col = 'user_id'
item_col = 'item_id'

In [ ]:
data.shape, data.user_id.nunique(), data.item_id.nunique()

In [ ]:
data[['user_id', 'item_id']].duplicated().sum()

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()

### Convert timestamp

In [ ]:
%%time 
# 5s
data['date'] = data['date'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d'))

In [ ]:
data.head()

In [ ]:
%%time
# 198ms
interactions_per_week = data.groupby(by=['week']).count().iloc[:, 0]
interactions_per_week.name = 'count'
interactions_per_week=interactions_per_week.reset_index()
_ = interactions_per_week.copy()
# _['date'] = _['date'].dt.date
_.groupby('week').sum().plot(kind='bar')
plt.title('interactions per week')

___
## Get intervals and Frequent users (threshold = 0.8)

time intervals

In [ ]:
# user_presence_df = pd.read_csv('output/amazonbooks_dump/3rd_sample_user_presence_df.csv')
# user_month_interactions = pd.read_csv('output/amazonbooks_dump/3rd_sample_user_month_interactions.csv')
frequent_users_month = joblib.load('output/amazonbooks_dump/3rd_sample_frequent_users_month.joblib')

___
## Experiments using months

In [ ]:
data.shape

In [ ]:
%%time
# 1min 26s
buckets, holdouts = getBucketsHoldouts(
    data=data, # data[ data['week'] == 1 ].loc[:20000],#
    user_col=user_col,
    item_col=item_col,
    frequent_users=data['user_id'].unique(), #frequent_users_month, # to use every user in holdout: data['user_id'].unique()
    interval_type='W',
    intervals=None, 
    cold_start_buckets=0)

In [ ]:
sum( [b.size for b in buckets] ) + sum( [h.size for h in holdouts] )

In [ ]:
joblib.dump(buckets, 'output/amazonbooks_dump/3rd_sample_buckets.joblib')
joblib.dump(holdouts, 'output/amazonbooks_dump/3rd_sample_holdouts.joblib')

### ISGD

In [ ]:
buckets = joblib.load('output/amazonbooks_dump/3rd_sample_buckets.joblib')
holdouts = joblib.load('output/amazonbooks_dump/3rd_sample_holdouts.joblib')

Hyperparameters

In [ ]:
%%time
import itertools
def grid_search(model, stream, random_seed = 10, interleaved=10):    
    num_factors = [100, 150, 200]
    num_iter = [2, 5, 8]
    learn_rate = [0.1, 0.25, 0.5]
    regularization = [0.3, 0.4, 0.5]
    num_nodes = [4, 8, 16]
    grid = [num_factors, num_iter, learn_rate, regularization, num_nodes]
    grid = list(itertools.product(*grid))
    results = []
    for i, hp in enumerate(grid):
        print(((i*100)/len(grid)), '%')
        empty_stream = ImplicitData([], [])
        nf, ni, lr, reg, nn = hp
        m = model(empty_stream, nf, ni, lr, reg, reg, random_seed)
        e = EvalPrequential(m, stream, metrics = ["Recall@N"])
        result = e.Evaluate(start_eval=0, count=stream.size, interleaved=interleaved)
        results.append( np.mean(result['Recall@N']) )
    return grid, results    
prop = 0.05
hp_sample = data.iloc[:round( data.shape[0]*prop )]
stream = ImplicitData(hp_sample[user_col], hp_sample[item_col]) #(data['playlist_id'],data['track_id'])    

# 6h 25min 45s
grid, results = grid_search(model=ISGD, stream=stream, random_seed=10, interleaved=1 )
len(grid), len(results), max(results) # (243, 243, 0.006805807622504537)

In [ ]:
# num_factors, num_iter, learn_rate, regularization, num_nodes = grid[ np.argmax( results ) ]
num_factors, num_iter, learn_rate, regularization, num_nodes = grid[ np.argmax( results ) ] # (100, 8, 0.5, 0.3, 4)

In [ ]:
# # define hyperparameters (SAME AS LASTFM)
# num_factors = 160
# num_iter = 4
# learn_rate = 0.5
# regularization = 0.4
# num_nodes = 8

In [ ]:
# transforma intera??es em objeto que contem mappings usu?rio-itens e item-usu?rios, cont?m tamb?m m?todos de suporte. recebe listas
# stream = ImplicitData(data[user_col], data[item_col])
# O modelo deve ser iniciado com uma lista vazia
empty_stream = ImplicitData([], [])
# Se o stream for passado, ao excluir itens conhecidos o recall ? sempre 0. Ao permitir a recomenda??o de itens j? vistos, o recall n?o ? 0.
model = ISGD(empty_stream, num_factors, num_iter, learn_rate = learn_rate, u_regularization = regularization, i_regularization = regularization, random_seed = 10)

In [ ]:
# criamos instancia de EvaluateHoldouts para treinar o modelo e criar checkpoints
eval = EvaluateHoldouts(model=model, buckets=buckets, holdouts=holdouts)

In [ ]:
%%time
# 4h 36min 18s
eval.Train_Evaluate(N_recommendations=20, exclude_known_items=True, default_user='none')

In [ ]:
rm = eval.results_matrix
df = pd.DataFrame(rm)
df.to_csv('output/amazonbooks_dump/3rd_sample_amazon_books month_bucket ISGD results.csv', index=False)
recall_heatmap(df, 
               round_point=3,
               title='Recall@20 for ISGD models across Holdouts - Amazon Books',
               filepath='images/heatmaps/amazonbooks_dump/3rd_sample_amazon_books month_bucket ISGD heatmap.png') #='images/heatmaps/palco_2010 month_bucket ISGD heatmap.png'

In [ ]:
arecall = avg_recall(df)
arecall

In [ ]:
BWT, meanBWT = compute_BWT(df)
BWT, meanBWT

In [ ]:
FWT = compute_FWT(df)
FWT
# que itens que usuario utilizou no passado e deixou de consumir o sistema ainda pode recomendar

In [ ]:
joblib.dump(eval.IncrementalTraining_time_record, 'output/amazonbooks_dump/3rd_sample_amazon_books month_bucket ISGD training time.joblib')
joblib.dump(eval.EvaluateHoldouts_time_record, 'output/amazonbooks_dump/3rd_sample_amazon_books month_bucket ISGD eval time.joblib')